## aws bedrock コマンド

コントロールプレーン。Bedrock の各機能を使用するために使用される。

In [ ]:
# 例: 東京リージョンで使用できるモデル一覧
!aws bedrock list-foundation-models --region ap-northeast-1 --query 'modelSummaries[*].[modelId,modelName]' --output table

In [ ]:
# 例: 東京リージョンで使用できる推論プロファイル一覧
!aws bedrock list-inference-profiles --region ap-northeast-1 --query 'inferenceProfileSummaries[*].[inferenceProfileName,inferenceProfileId]' --output table

In [ ]:
# 例: Bedrock ガードレールを作成
!aws bedrock create-guardrail \
  --name "my-guardrail" \
  --description "My guardrail description" \
  --blocked-input-messaging "申し訳ございません。この入力は処理できません。" \
  --blocked-outputs-messaging "申し訳ございません。この出力は提供できません。" \
  --content-policy-config 'filtersConfig=[{type=VIOLENCE,inputStrength=MEDIUM,outputStrength=MEDIUM}]' \
  --region ap-northeast-1

In [ ]:
# 例: Bedrock ガードレールを削除
!aws bedrock delete-guardrail \
  --guardrail-identifier <guardrail-id> \
  --region ap-northeast-1

## aws bedrock-runtime コマンド

データプレーン。基盤モデルとのやりとりに使用される。

In [ ]:
# 例: モデルの呼び出し (invoke-model)
!aws bedrock-runtime invoke-model \
  --model-id jp.amazon.nova-2-lite-v1:0 \
  --body '{
    "messages": [{"role": "user", "content": [{"text": "こんにちは"}]}],
    "inferenceConfig": {"maxTokens": 500, "temperature": 0.7}
  }' \
  --region ap-northeast-1 \
  output.txt

!cat output.txt && rm output.txt

In [ ]:
# 例: モデルの呼び出し (converse)
!aws bedrock-runtime converse \
  --model-id jp.amazon.nova-2-lite-v1:0 \
  --messages '[{"role": "user", "content": [{"text": "こんにちは"}]}]' \
  --inference-config '{"maxTokens":500, "temperature":0.7}' \
  --additional-model-request-fields '{}' \
  --region ap-northeast-1

なお、invoke-model がモデルごとに異なるフォーマットの body を指定する必要があるのに対して、converse は同じフォーマットで異なるモデルを呼び出せるように工夫されています（モデル固有の引数は additional-model-request-fields にて指定）。

そのため、invoke-model ではなく converse の利用が推奨されています。

https://docs.aws.amazon.com/ja_jp/bedrock/latest/userguide/conversation-inference.html
```
メッセージをサポートするすべての Amazon Bedrock モデルで動作する一貫した API を提供する Converse API を使用することをお勧めします。そうすることで、コードを 1 回だけ記述し、それをさまざまなモデルで使用できます。
```